# Title : Predicting Airbnb Listing Prices with MLflow and AWS S3
#### Name : Yash Suthar

## 1. Load the Data

In [1]:
import s3fs
import pandas as pd
import numpy as np


# Create an S3 file system object
fs = s3fs.S3FileSystem()

# Method 1: Read CSV directly with pandas
bucket_name = "c0957228"
file_key = "AB_NYC_2019.csv"

# Read CSV file directly from S3
df = pd.read_csv(fs.open(f"{bucket_name}/{file_key}"))
print(df.head())

     id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name neighbourhood_group neighbourhood  latitude  longitude  \
0         John            Brooklyn    Kensington  40.64749  -73.97237   
1     Jennifer           Manhattan       Midtown  40.75362  -73.98377   
2    Elisabeth           Manhattan        Harlem  40.80902  -73.94190   
3  LisaRoxanne            Brooklyn  Clinton Hill  40.68514  -73.95976   
4        Laura           Manhattan   East Harlem  40.79851  -73.94399   

         room_type  price  minimum_nights  number_of_reviews last_review  \
0     Private room    149               1                  9  20

## 2. DROP IRRELEVANT COLUMNS

In [2]:
# IDs and text columns not useful for ML
df.drop(columns=['id', 'name', 'host_name'], inplace=True)

## 3. HANDLE MISSING VALUES

In [3]:
# reviews_per_month → fill with 0 (no reviews)
df['reviews_per_month'].fillna(0, inplace=True)

# last_review → convert to datetime first
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

# create new feature: days_since_last_review
df['days_since_last_review'] = (pd.Timestamp("2019-12-31") - df['last_review']).dt.days

# fill missing dates with large value (no recent reviews)
df['days_since_last_review'].fillna(999, inplace=True)

# drop original column
df.drop(columns=['last_review'], inplace=True)

C:\Users\yashs\AppData\Local\Temp\ipykernel_5052\3493860602.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['reviews_per_month'].fillna(0, inplace=True)
C:\Users\yashs\AppData\Local\Temp\ipykernel_5052\3493860602.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

## 4. HANDLE OUTLIERS

In [4]:
# Price filtering (remove extreme outliers)
df = df[(df['price'] > 0) & (df['price'] < 1000)]

# Minimum nights (remove extreme unrealistic values)
df = df[df['minimum_nights'] < 365]

## 5. FEATURE ENGINEERING

In [5]:
# log transform target (very important for price skewness)
df['price'] = np.log1p(df['price'])

## 6. ENCODE CATEGORICAL FEATURES

In [6]:
# One-hot encoding
df = pd.get_dummies(df, columns=[
    'neighbourhood_group',
    'neighbourhood',
    'room_type'
], drop_first=True)

## 7. Setup MLflow

In [7]:
import mlflow
import mlflow.sklearn

In [8]:
# ===============================
# FIX REMAINING NaN VALUES
# ===============================

# Check NaNs
print(df.isnull().sum())

# Fill all remaining NaNs
df.fillna(0, inplace=True)

host_id                   0
latitude                  0
longitude                 0
price                     0
minimum_nights            0
                         ..
neighbourhood_Woodlawn    0
neighbourhood_Woodrow     0
neighbourhood_Woodside    0
room_type_Private room    0
room_type_Shared room     0
Length: 236, dtype: int64


## 8. Train-Test Split + Scaling

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 9. 3 Regression Models

I am using
- Linear Regrssion
- Ranom Forest
- Gradient Boosting

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

## 10. Evaluation Function

In [11]:
from sklearn.metrics import mean_squared_error, r2_score

def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    return rmse, r2

## 11. MLflow Experiment Tracking

In [12]:
mlflow.set_experiment("Airbnb Price Prediction")

2026/04/01 22:56:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/01 22:56:01 INFO mlflow.store.db.utils: Updating database tables
2026/04/01 22:56:02 INFO mlflow.tracking.fluent: Experiment with name 'Airbnb Price Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/yashs/Desktop/softwaretools_and_emerging/Assignment-2/predicting-airbnb-price/mlruns/1', creation_time=1775098562008, experiment_id='1', last_update_time=1775098562008, lifecycle_stage='active', name='Airbnb Price Prediction', tags={}, workspace='default'>

## 12. Train + Log Everything

In [13]:
best_rmse = float("inf")
best_model_name = None
best_model = None

for name, model in models.items():
    
    with mlflow.start_run(run_name=name):
        
        # Train
        model.fit(X_train, y_train)
        
        # Evaluate
        rmse, r2 = evaluate_model(model, X_test, y_test)
        
        # Log parameters
        mlflow.log_param("model_name", name)
        
        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)
        
        # Log metrics
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2_score", r2)
        
        # Log model
        mlflow.sklearn.log_model(model, name)
        
        print(f"{name} → RMSE: {rmse}, R2: {r2}")
        
        # Track best model
        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model
            best_model_name = name

2026/04/01 22:56:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 22:56:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearRegression → RMSE: 0.42709678520958283, R2: 0.5804700536533063


2026/04/01 22:56:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 22:56:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest → RMSE: 0.39518892695063723, R2: 0.6408135902883643


2026/04/01 22:57:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 22:57:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


GradientBoosting → RMSE: 0.4069428469954591, R2: 0.6191296194967979


## 13. Register Best Model

In [14]:
mlflow.sklearn.log_model(
    best_model,
    "best_model",
    registered_model_name="AirbnbPriceModel"
)

2026/04/01 22:57:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 22:57:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'AirbnbPriceModel'.
Created version '1' of model 'AirbnbPriceModel'.
